# ClearView · Entrenar el detector de publicidad

Este cuaderno entrena el detector que usa ClearView (web del móvil y Quest 3): **vallas, carteles y logos**
como publicidad, y **personas, vehículos, sillas de ruedas, semáforos y señales** como objetos que nunca se tapan.

**Antes de empezar:** menú *Entorno de ejecución → Cambiar tipo de entorno de ejecución → GPU T4*.
Después, *Entorno de ejecución → Ejecutar todo*. Tarda unas 3–4 horas en total.

| Paso | Qué hace | Tiempo aprox. |
|---|---|---|
| 1 | Prepara el entorno y descarga el código | 2 min |
| 2 | Ajustes | — |
| 3 | Descarga vallas, carteles, personas, vehículos y señales de Open Images (sin cuenta) | 15–25 min |
| 4 | Descarga logos (LogoDet-3K desde Kaggle, Roboflow o un zip tuyo) | 5–15 min |
| 5 | Añade personas y vehículos que falten en las fotos (autoetiquetado) | 15–30 min |
| 6 | Une todo en un dataset | 2 min |
| 7 | Entrena YOLO11n | 2–3 h |
| 8 | Evalúa, sobre todo la seguridad | 2 min |
| 9 | Exporta para la web y para Unity, y descarga el resultado | 3 min |

Si Colab se desconecta, vuelve a ejecutar todo: con Google Drive activado (paso 1) el entrenamiento
continúa donde se quedó.

## 1 · Entorno

In [ ]:
#@title Preparar entorno { display-mode: "form" }
USAR_DRIVE = True  #@param {type:"boolean"}
RAMA = "ccr-3d807d19-g1sbwv"  #@param {type:"string"}

import os, subprocess
gpu = subprocess.run(["nvidia-smi", "-L"], capture_output=True, text=True).stdout.strip()
print(gpu or "AVISO: no hay GPU. Cambia el tipo de entorno a GPU T4 o el entrenamiento tardará días.")

if USAR_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    SALIDA = "/content/drive/MyDrive/clearview"
else:
    SALIDA = "/content/clearview_salida"
os.makedirs(SALIDA, exist_ok=True)

if not os.path.isdir("/content/clearview"):
    !git clone -q -b {RAMA} https://github.com/jmendozapuche/clearview.git /content/clearview
%cd /content/clearview
!git pull -q
!pip -q install -r ml/requirements.txt
print("Listo. Resultados en", SALIDA)

## 2 · Ajustes
Los valores por defecto están pensados para una T4 gratuita. Más imágenes y épocas dan mejor modelo, pero tardan más.

In [ ]:
#@title Ajustes { display-mode: "form" }
IMAGENES_POR_CLASE_ANUNCIO = 4000  #@param {type:"integer"}
IMAGENES_POR_CLASE_SEGURIDAD = 2500  #@param {type:"integer"}
FUENTE_LOGOS = "kaggle"  #@param ["kaggle", "roboflow", "zip", "ninguna"]
DATASET_KAGGLE = "lyly99/logodet3k"  #@param {type:"string"}
MAX_IMAGENES_LOGOS = 8000  #@param {type:"integer"}
EPOCAS = 80  #@param {type:"integer"}
TAMANO = 416  #@param {type:"integer"}

#@markdown **Personas impresas en anuncios.** En Open Images, el 51 % de las personas que aparecen en fotos de
#@markdown vallas y carteles son *representaciones* (fotos o dibujos dentro del anuncio). Si se mantienen como
#@markdown "persona", ClearView deja visibles esos anuncios (seguro, pero oculta menos). Si se quitan, el
#@markdown modelo aprende a distinguirlas y oculta más anuncios, **pero solo es aceptable si el paso 8
#@markdown confirma que la detección de personas reales sigue alta**.
QUITAR_PERSONAS_IMPRESAS = False  #@param {type:"boolean"}

## 3 · Open Images: vallas, carteles y objetos de seguridad

In [ ]:
extra = "--drop-safety-depictions" if QUITAR_PERSONAS_IMPRESAS else ""
!python ml/scripts/fetch_openimages.py --split train --out raw/openimages --cache raw/openimages_cache \
    --ads-per-class {IMAGENES_POR_CLASE_ANUNCIO} --safety-per-class {IMAGENES_POR_CLASE_SEGURIDAD} --workers 48 {extra}

## 4 · Logos
- **kaggle** (recomendado): LogoDet-3K, unas 3.000 marcas. Necesita tu clave de Kaggle: en kaggle.com → *Settings → API → Create New Token*
  descarga `kaggle.json`. Guárdala en Colab como *secretos* (icono de llave, a la izquierda) `KAGGLE_USERNAME` y `KAGGLE_KEY`,
  o súbela cuando la celda te lo pida. Si el dataset no se encuentra, busca "LogoDet-3K" en Kaggle y copia su identificador
  (`usuario/nombre`) en el ajuste `DATASET_KAGGLE`.
- **roboflow**: un dataset de logos de Roboflow Universe. Pide tu clave (`ROBOFLOW_API_KEY` en secretos) y el proyecto.
- **zip**: sube un zip con imágenes y etiquetas en formato YOLO o Pascal VOC.
- **ninguna**: entrena sin logos (solo vallas y carteles).

In [ ]:
import os, glob, json, shutil, zipfile
from pathlib import Path

def secreto(nombre):
    try:
        from google.colab import userdata
        return userdata.get(nombre)
    except Exception:
        return os.environ.get(nombre)

def a_yolo_logo(origen):
    """Convierte lo descargado (VOC o YOLO) a raw/logos con una sola clase 'logo'."""
    shutil.rmtree("raw/logos", ignore_errors=True)
    if glob.glob(f"{origen}/**/*.xml", recursive=True):
        !python ml/scripts/voc_to_yolo.py --src {origen} --out raw/logos --single-class logo --max-images {MAX_IMAGENES_LOGOS}
    else:
        !python ml/scripts/normalize_yolo.py --src {origen} --out raw/logos --single-class logo --max-images {MAX_IMAGENES_LOGOS}

if FUENTE_LOGOS == "kaggle":
    usuario, clave = secreto("KAGGLE_USERNAME"), secreto("KAGGLE_KEY")
    if not (usuario and clave):
        from google.colab import files
        print("Sube tu kaggle.json")
        subido = files.upload()
        datos = json.loads(next(iter(subido.values())))
        usuario, clave = datos["username"], datos["key"]
    os.environ["KAGGLE_USERNAME"], os.environ["KAGGLE_KEY"] = usuario, clave
    !pip -q install kaggle
    r = !kaggle datasets download -d {DATASET_KAGGLE} -p raw/logos_descarga --unzip 2>&1
    print("\n".join(r[-5:]))
    if not glob.glob("raw/logos_descarga/**/*.*", recursive=True):
        raise SystemExit(f"No se pudo descargar '{DATASET_KAGGLE}'. Revisa el identificador en Kaggle y tu clave.")
    a_yolo_logo("raw/logos_descarga")

elif FUENTE_LOGOS == "roboflow":
    #@markdown Para Roboflow: workspace, proyecto y versión del dataset (aparecen en su URL de Universe).
    RF_WORKSPACE = ""  #@param {type:"string"}
    RF_PROYECTO = ""  #@param {type:"string"}
    RF_VERSION = 1  #@param {type:"integer"}
    !pip -q install roboflow
    from roboflow import Roboflow
    clave = secreto("ROBOFLOW_API_KEY") or input("Clave de API de Roboflow: ")
    Roboflow(api_key=clave).workspace(RF_WORKSPACE).project(RF_PROYECTO).version(RF_VERSION).download(
        "yolov8", location="raw/logos_descarga")
    a_yolo_logo("raw/logos_descarga")

elif FUENTE_LOGOS == "zip":
    from google.colab import files
    print("Sube un .zip con imágenes y etiquetas (YOLO o Pascal VOC)")
    subido = files.upload()
    for nombre in subido:
        zipfile.ZipFile(nombre).extractall("raw/logos_descarga")
    a_yolo_logo("raw/logos_descarga")
else:
    print("Sin logos: el modelo aprenderá vallas y carteles.")

## 5 · Autoetiquetado de seguridad
Los datasets de logos no marcan a las personas ni los vehículos que salen en sus fotos. Sin este paso, el modelo
aprendería que una persona en esas fotos es "fondo", y detectaría peor justo lo que nunca hay que tapar.
Un modelo grande (YOLO11x) las marca automáticamente, sin duplicar las que ya estaban etiquetadas.

In [ ]:
if os.path.isdir("raw/logos/images"):
    !python ml/scripts/pseudo_label_safety.py --source raw/logos --model yolo11x.pt --imgsz 640
if not QUITAR_PERSONAS_IMPRESAS:
    # Open Images no siempre marca todas las personas; completarlas refuerza la seguridad.
    # (Si se quitan las personas impresas, este paso se salta: YOLO11x las volvería a marcar.)
    !python ml/scripts/pseudo_label_safety.py --source raw/openimages --model yolo11x.pt --imgsz 640

## 6 · Unir los datos

In [ ]:
!python ml/scripts/prepare_dataset.py --config ml/configs/colab.yaml

## 7 · Entrenar
Guarda el progreso en Drive cada época; si se corta, vuelve a ejecutar y continúa.

In [ ]:
!python ml/scripts/train.py --data datasets/clearview/data.yaml --model yolo11n.pt --imgsz {TAMANO} \
    --epochs {EPOCAS} --batch -1 --device 0 --project {SALIDA}/runs --name clearview --resume

## 8 · Evaluar (la seguridad primero)
Mira sobre todo la columna **recall** de las clases de **seguridad**: es la parte de personas, vehículos y señales
reales que el modelo detecta. Si alguna sale marcada, el modelo no es seguro todavía: más imágenes de esa clase o más épocas.

In [ ]:
PESOS = f"{SALIDA}/runs/clearview/weights/best.pt"
!python ml/scripts/eval_safety.py --weights {PESOS} --data datasets/clearview/data.yaml --config ml/configs/colab.yaml --imgsz {TAMANO} --device 0

## 9 · Exportar y descargar

In [ ]:
shutil.rmtree("exportado", ignore_errors=True)
!python ml/scripts/export.py --weights {PESOS} --config ml/configs/colab.yaml --unity exportado/unity --web exportado/web
!cp ml/configs/colab.yaml exportado/
shutil.make_archive(f"{SALIDA}/clearview_modelo", "zip", "exportado")
print("Guardado en", f"{SALIDA}/clearview_modelo.zip")
from google.colab import files
files.download(f"{SALIDA}/clearview_modelo.zip")

## Siguiente paso
Sube `clearview_modelo.zip` a tu repositorio en GitHub: abre el repo, cambia a la rama `ccr-3d807d19-g1sbwv`,
*Add file → Upload files*, arrastra el zip y pulsa *Commit changes*. Después dile a Claude que ya está:
lo instalará en la web y en el proyecto de Unity.